In [0]:
%python
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score

pdf = spark.table("clinical_portfolio.gold.model_table").toPandas()
y = pdf.pop("label")
X = pdf.drop(columns=["pt_id", "outcome_group"])
num = list(X.select_dtypes("number").columns)
cat = [c for c in X.columns if c not in num]
X[cat] = X[cat].astype(object).where(X[cat].notna(), None)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])

# The test set is touched once, at the very end of the comparison.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [0]:
%python
version = spark.sql("DESCRIBE HISTORY clinical_portfolio.gold.model_table LIMIT 1").first()["version"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

candidates = {
    "dummy_prior":   DummyClassifier(strategy="prior"),
    "logistic":      LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(300, min_samples_leaf=5,
                                            class_weight="balanced", random_state=0),
}

for name, estimator in candidates.items():
    pipe = Pipeline([("pre", pre), ("model", estimator)])
    with mlflow.start_run(run_name=name):
        scores = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc")
        pipe.fit(Xtr, ytr)
        p = pipe.predict_proba(Xte)[:, 1]
        mlflow.log_param("model", name)
        mlflow.log_param("gold_table_version", version)
        mlflow.log_metric("cv_auc_mean", scores.mean())
        mlflow.log_metric("cv_auc_std", scores.std())
        mlflow.log_metric("test_auc", roc_auc_score(yte, p))
        mlflow.log_metric("test_pr_auc", average_precision_score(yte, p))
        mlflow.sklearn.log_model(
            pipe, name="model",
            signature=infer_signature(Xtr, pipe.predict_proba(Xtr)[:, 1]),
            input_example=Xtr.head(3),
            serialization_format="cloudpickle")

In [0]:
%python
mlflow.set_registry_uri("databricks-uc")
best = mlflow.search_runs(order_by=["metrics.cv_auc_mean DESC"]).iloc[0]
name = "clinical_portfolio.gold.tb_outcome_model"
mv = mlflow.register_model(f"runs:/{best.run_id}/model", name)
mlflow.MlflowClient().set_registered_model_alias(name, "champion", mv.version)

In [0]:
%python
import joblib
best_pipe = mlflow.sklearn.load_model("models:/clinical_portfolio.gold.tb_outcome_model@champion")
defaults = X.iloc[[0]].copy()
for c in num: defaults[c] = float(X[c].median())
for c in cat: defaults[c] = X[c].mode(dropna=True).iloc[0]
joblib.dump({"model": best_pipe, "defaults": defaults},
            "/Workspace/Users/alliesdl@gmail.com/MLops_bal_pipeline/app/model.joblib")

In [0]:
%python
runs = mlflow.search_runs(order_by=["metrics.cv_auc_mean DESC"])
cols = ["tags.mlflow.runName", "metrics.cv_auc_mean", "metrics.cv_auc_std",
        "metrics.test_auc", "metrics.test_pr_auc"]
display(runs[cols].round(3))